# PySpark 5: writing, pipelines and quality

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-05-writing-pipelines-and-quality.ipynb)

Your order pipeline wrote its output, then the scheduler lost the success acknowledgement. It will run the same batch again. Which rows can change, and what should remain identical?

Follow a batch from validation and quarantine through the write. Compare a bounded partition replacement with a key-based merge, and use small tests to pin down the behavior of each transformation. The local Delta lab runs actual table mutations.

**Check your work:** name the replacement scope and record key, then test a duplicate delivery and a stale update. The late-update scenario explains the expected result before you write the merge.

## Setup and sample data

This module runs on the same four tables as PySpark 1, so each module can run on its own. The setup cell installs PySpark when it is missing and starts a local session.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {'pyspark': 'pyspark>=4.0,<4.1', 'deltalake': 'deltalake>=1.0', 'pyarrow': 'pyarrow>=15'}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

# Python workers must use this notebook's interpreter, and every timestamp in the lessons is UTC.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["TZ"] = "UTC"
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-lessons-")

spark = (
    SparkSession.builder
    .appName("pyspark-lessons")
    .master("local[2]")  # driver and executor share one JVM: the development topology
    .config("spark.sql.shuffle.partitions", "4")  # 200 is the default; tiny data needs few
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

## Inputs for this notebook

The order and item fixtures support the write and quality examples. The Delta lab creates its own local table so it can run independently of an external catalog.

In [ ]:
from datetime import date, datetime
from decimal import Decimal as D

from pyspark.sql import types as T

customers_schema = T.StructType([
    T.StructField("customer_id", T.IntegerType(), False),
    T.StructField("name", T.StringType(), False),
    T.StructField("region", T.StringType(), True),
    T.StructField("signup_date", T.DateType(), False),
])

customers = spark.createDataFrame(
    [
        (1, "Acme Corp",      "North", date(2024, 1, 3)),
        (2, "Bright Retail",  "North", date(2024, 2, 11)),
        (3, "Cyan Traders",   "South", date(2024, 3, 15)),
        (4, "Delta Grocers",  None,    date(2024, 5, 2)),
        (5, "Echo Supplies",  "West",  date(2024, 6, 20)),
        (6, "Foxtrot Foods",  "West",  date(2024, 8, 1)),
        (7, "Gamma Sports",   "East",  date(2024, 9, 14)),
        (8, "Hotel Homeware", "East",  date(2024, 11, 5)),
    ],
    schema=customers_schema,
)

customers.orderBy("customer_id").show()

In [ ]:
from pyspark.sql import types as T

orders_schema = T.StructType([
    T.StructField("order_id", T.IntegerType(), False),
    T.StructField("customer_id", T.IntegerType(), False),
    T.StructField("status", T.StringType(), True),
    T.StructField("order_ts", T.TimestampType(), False),
    T.StructField("amount", T.DecimalType(12, 2), True),
])

orders = spark.createDataFrame(
    [
        (1,  1, "paid",     datetime(2026, 1,  5,  9, 15), D("120.00")),
        (2,  1, "shipped",  datetime(2026, 1,  5, 10,  5),  D("84.50")),
        (3,  1, "returned", datetime(2026, 1,  5, 11, 40),  D("30.00")),
        (4,  2, "paid",     datetime(2026, 1,  6, 15, 20), D("260.00")),
        (5,  2, "placed",   datetime(2026, 1,  7,  8,  5),  D("55.25")),
        (6,  3, "shipped",  datetime(2026, 1,  7, 17, 45), D("149.00")),
        (7,  3, "paid",     datetime(2026, 1,  9, 12, 10), D("310.00")),
        (8,  4, "returned", datetime(2026, 1, 10, 19, 30),  D("89.90")),
        (9,  4, "shipped",  datetime(2026, 1, 12,  8, 55), D("1180.00")),
        (10, 5, "paid",     datetime(2026, 1, 12, 13,  0),   D("74.00")),
        (11, 6, "placed",   datetime(2026, 1, 13, 10, 25),  D("210.00")),
        (12, 6, "shipped",  datetime(2026, 1, 14,  0, 30),   D("96.40")),
    ],
    schema=orders_schema,
)

orders.orderBy("order_id").show()
orders.printSchema()

In [ ]:
from pyspark.sql import types as T

order_items_schema = T.StructType([
    T.StructField("order_id", T.IntegerType(), False),
    T.StructField("sku", T.StringType(), False),
    T.StructField("qty", T.IntegerType(), False),
    T.StructField("unit_price", T.DecimalType(8, 2), False),
])

order_items = spark.createDataFrame(
    [
        (1,  "SKU-4001",  2,  D("12.50")),
        (1,  "SKU-4007",  1,  D("95.00")),
        (2,  "SKU-4002",  1,  D("34.50")),
        (2,  "SKU-4009",  2,  D("25.00")),
        (3,  "SKU-4002",  1,  D("30.00")),
        (4,  "SKU-4003",  1, D("149.00")),
        (4,  "SKU-4004",  2,  D("55.50")),
        (5,  "SKU-4005",  1,  D("55.25")),
        (6,  "SKU-4003",  1, D("149.00")),
        (7,  "SKU-4006",  1, D("189.00")),
        (7,  "SKU-4008",  1, D("121.00")),
        (8,  "SKU-4010",  1,  D("89.90")),
        (9,  "SKU-4011", 20,  D("59.00")),
        (10, "SKU-4012",  2,  D("37.00")),
        (11, "SKU-4013",  1, D("210.00")),
        (12, "SKU-4014",  1,  D("41.40")),
        (12, "SKU-4005",  1,  D("55.00")),
    ],
    schema=order_items_schema,
)

order_items.orderBy("order_id", "sku").show(20)

In [ ]:
from pyspark.sql import Window, functions as F

In [ ]:
import contextlib
import io


def plan_of(df, mode="formatted"):
    """Return the text `df.explain(mode)` prints."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain(mode)
    return buffer.getvalue()

In [ ]:
raw_customers = spark.createDataFrame(
    [
        ("  Ada@Example.COM ", " us", "2026-01-03 09:00:00", "+1 (312) 555-0100"),
        ("grace@example.org", "gb ", "2026-01-04 10:30:00", "020 7946 0018"),
        ("not-an-email", "us", "2026-01-05 11:00:00", None),
    ],
    "email string, country_code string, created_at string, phone string",
)

clean_customers = (
    raw_customers
    .select(
        F.trim(F.lower("email")).alias("email"),
        F.upper(F.trim("country_code")).alias("country_code"),
        F.to_timestamp("created_at").alias("created_at"),
        F.regexp_replace("phone", r"[^0-9]", "").alias("phone_digits"),
    )
    .withColumn("email_domain", F.element_at(F.split("email", "@"), -1))
    .where(F.col("email").contains("@"))
)
clean_customers.show(truncate=False)

rows = {r["email"]: r for r in clean_customers.collect()}
assert set(rows) == {"ada@example.com", "grace@example.org"}
assert rows["ada@example.com"]["phone_digits"] == "13125550100"
assert rows["grace@example.org"]["country_code"] == "GB"

The recipes in the earlier modules assume someone handed you clean tables. A production job asks who
wrote them and how: governed writes, incremental loads, tests, bad records, streaming,
monitoring, and the tuning levers that keep a nightly job boring.

## Choose the write scope before overwriting

With Unity Catalog, prefer table names like `catalog.schema.table` over raw paths in
application code, so governance, lineage, and permissions live at the table layer. The local
catalog has two levels (`schema.table`) but the habit is the same:

In [ ]:
daily_product_sales = (
    order_items
    .join(orders.select("order_id", "order_ts"), "order_id")
    .withColumn("order_date", F.to_date("order_ts"))
    .groupBy("order_date", "sku")
    .agg(F.sum("qty").alias("units"), F.sum(F.col("qty") * F.col("unit_price")).alias("revenue"))
)

spark.sql("create database if not exists gold")
daily_product_sales.write.mode("overwrite").saveAsTable("gold.daily_product_sales")
assert spark.table("gold.daily_product_sales").count() == daily_product_sales.count()

For partitioned tables add `partitionBy` (on Databricks add `.format("delta")`), and append
to an existing table with `mode("append")`. `df.writeTo("catalog.schema.table").append()` is the
v2 API for Delta and Iceberg catalogs; the local Parquet catalog only supports the v1 writer.

In [ ]:
(
    daily_product_sales
    .write
    .mode("overwrite")
    .partitionBy("order_date")
    .saveAsTable("gold.daily_product_sales_by_day")
)
partitions = spark.sql("show partitions gold.daily_product_sales_by_day").count()
print("partitions:", partitions)
assert partitions == daily_product_sales.select("order_date").distinct().count()

**How it works.** `saveAsTable` goes through the catalog, which owns the schema, permissions and lineage. `writeTo(...).append()` works only when the table already exists and uses the table's stored schema and partitioning. On a partitioned table, `mode("overwrite")` replaces the whole table unless `spark.sql.sources.partitionOverwriteMode` is `dynamic`, in which case only partitions present in the DataFrame are replaced.

Dynamic overwrite in practice: a pricing bug on 12 January is fixed, and only that day is
rewritten - the other days keep their files:

In [ ]:
sales_path = os.path.join(WORK_DIR, "daily_product_sales")
daily_product_sales.write.mode("overwrite").partitionBy("order_date").parquet(sales_path)
other_days_before = spark.read.parquet(sales_path).where("order_date != '2026-01-12'").agg(F.sum("revenue")).first()[0]

corrected = (
    daily_product_sales
    .where("order_date = '2026-01-12'")
    .withColumn("revenue", F.col("revenue") * F.lit(0.9))  # the reprocessed day
)
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
corrected.write.mode("overwrite").partitionBy("order_date").parquet(sales_path)
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "static")

after = spark.read.parquet(sales_path)
other_days_after = after.where("order_date != '2026-01-12'").agg(F.sum("revenue")).first()[0]
assert after.select("order_date").distinct().count() == daily_product_sales.select("order_date").distinct().count()
assert other_days_after == other_days_before
print("only 2026-01-12 was rewritten; the other days are untouched")

**Example.** A table partitioned by `order_date` with 365 days and one new day of data: a static overwrite rewrites all 365 partitions, a dynamic overwrite touches only the one partition present in the frame.

**Watch out for.** leaving `overwriteSchema` on while an upstream type drifts. The write succeeds and quietly changes the table contract; own the schema in the pipeline and fail fast instead.

## Process only the changes in the current delivery

An incremental pipeline processes only new or changed records. The usual pattern is to use a watermark, ingestion timestamp, source sequence number, or table change feed.

A batch watermark lives in an ops table. Read it, take only newer bronze rows, keep the latest
change per key, then upsert. Delta and Iceberg provide `MERGE`:

```sql
merge into main.silver.orders as target
using staged_orders as source
on target.order_id = source.order_id
when matched then update set *
when not matched then insert *
```

Plain Parquet has no `MERGE`, so the cell below expresses the same upsert as "rows whose key did
not change, plus the staged rows" - and runs it twice to prove it is idempotent.

In [ ]:
watermarks = spark.createDataFrame(
    [("orders_silver", datetime(2026, 1, 10, 0, 0))],
    "pipeline_name string, loaded_until_ts timestamp",
)
last_loaded_ts = (
    watermarks.where(F.col("pipeline_name") == "orders_silver")
    .select(F.max("loaded_until_ts").alias("ts"))
    .first()["ts"]
)

silver = orders.where(F.col("order_ts") <= F.lit(last_loaded_ts))  # what the last run loaded

bronze_changes = (
    orders.where(F.col("order_ts") > F.lit(last_loaded_ts))
    .withColumn("source_lsn", F.lit(1))
    .withColumn("ingested_at", F.col("order_ts"))
    .unionByName(spark.createDataFrame(  # order 8 was refunded, and the change arrived twice
        [(8, 4, "refunded", datetime(2026, 1, 10, 19, 30), D("89.90"), 2, datetime(2026, 1, 15, 8, 0)),
         (8, 4, "refunded", datetime(2026, 1, 10, 19, 30), D("89.90"), 2, datetime(2026, 1, 15, 8, 5))],
        T.StructType(silver.schema.fields + [T.StructField("source_lsn", T.IntegerType()), T.StructField("ingested_at", T.TimestampType())]),
    ))
)

staging_window = Window.partitionBy("order_id").orderBy(F.col("source_lsn").desc(), F.col("ingested_at").desc())
staged = (
    bronze_changes.withColumn("rn", F.row_number().over(staging_window))
    .where("rn = 1")
    .drop("rn", "source_lsn", "ingested_at")
)


def upsert(target, changes, key="order_id"):
    untouched = target.join(changes.select(key), key, "left_anti")
    return untouched.unionByName(changes)


once = upsert(silver, staged)
twice = upsert(once, staged)

assert once.count() == orders.count()
assert once.exceptAll(twice).count() == 0 and twice.exceptAll(once).count() == 0
assert once.where("order_id = 8").first()["status"] == "refunded"
print("rerunning the same batch leaves silver unchanged")

Interview point: a reliable incremental job needs idempotency. If the job retries the same batch, the target should end in the same state, not duplicate data.

**How it works.** The watermark read (`max(loaded_until_ts)`) and the `MERGE` are separate actions: the first is a small job that returns one row to the driver, the second is a full job. Delta and Iceberg implement `MERGE` by rewriting the files that contain matched keys and committing a new table version, so idempotency comes from matching on the business key, not from row-by-row updates.

**Example.** A batch of 5 million changed orders merging into a 2 billion row table: with partition pruning on `order_date`, `MERGE` rewrites only the files in the touched partitions rather than the whole table. The job takes minutes instead of hours.

**Watch out for.** deriving the watermark from `max(ingested_at)` on the target itself. Late data that arrives after that value is skipped forever; keep the watermark in an ops table and use a deliberate lookback window.

## Test a business rule with small inputs

Keep business transformations as functions that accept and return DataFrames. Then create small test DataFrames and compare outputs.

In [ ]:
def add_order_date(df):
    return df.withColumn("order_date", F.to_date("order_ts"))


def test_add_order_date():
    input_df = spark.createDataFrame([("o1", datetime(2026, 10, 4, 12, 30, 0))], ["order_id", "order_ts"])
    actual = add_order_date(input_df).select("order_id", "order_date").collect()
    assert str(actual[0]["order_date"]) == "2026-10-04"


def test_add_order_date_keeps_nulls():
    input_df = spark.createDataFrame(
        [("o1", "2026-10-04 12:30:00"), ("o2", None)],
        "order_id string, order_ts string",
    )
    rows = add_order_date(input_df).orderBy("order_id").collect()
    assert [r["order_id"] for r in rows] == ["o1", "o2"]
    assert str(rows[0]["order_date"]) == "2026-10-04"
    assert rows[1]["order_date"] is None


for test in (test_add_order_date, test_add_order_date_keeps_nulls):
    test()
    print("passed:", test.__name__)

For real projects, use `pytest`, fixtures for the Spark session, and small deterministic inputs. Test data should cover nulls, duplicate keys, late-arriving data, invalid records, and boundary timestamps.

**How it works.** A DataFrame test needs a session, but `createDataFrame` from local rows builds a small deterministic frame with no cluster involved. The test should compare collected rows in a defined order and also assert on schema and nullability, because a transformation that changes types is as much a bug as one that changes values.

**Example.** The one-row test above builds a one-partition local frame, runs a single action and finishes in milliseconds, so a suite of them fits in a normal CI run against one shared local `SparkSession` fixture.

**Watch out for.** asserting on `collect()` order without an `orderBy`. Spark does not guarantee row order, so the test passes locally and fails in CI for no real reason.

## Quarantine bad records and account for them

Bad records should be isolated, measurable, and visible. Do not silently drop them unless the product owner agrees that loss is acceptable.

In [ ]:
bronze_orders_raw = spark.createDataFrame(
    [(1, 10, D("50.00")), (2, None, D("20.00")), (None, 11, D("5.00")), (4, 12, D("-3.00")), (5, 13, D("8.00"))],
    "order_id int, customer_id int, amount decimal(12,2)",
)

checked = bronze_orders_raw.withColumn(
    "quality_error",
    F.when(F.col("order_id").isNull(), "missing_order_id")
     .when(F.col("customer_id").isNull(), "missing_customer_id")
     .when(F.col("amount") < 0, "negative_amount"),
).cache()  # both writes below read the same checked frame

valid_orders = checked.where("quality_error is null").drop("quality_error")
quarantine = checked.where("quality_error is not null").withColumn("rejected_at", F.current_timestamp())

valid_orders.write.mode("overwrite").saveAsTable("silver_orders_checked")
quarantine.write.mode("overwrite").saveAsTable("ops_orders_quarantine")

quarantine.groupBy("quality_error").count().orderBy("quality_error").show()
assert valid_orders.count() == 2 and quarantine.count() == 3
checked.unpersist()

Interview point: quality checks should happen before publishing gold tables. A dashboard should not discover broken data before the pipeline does.

**How it works.** A rule column evaluated once gives one plan with two filters over the same scan, so both the valid and the quarantine write read the source. `subtract` (and `exceptAll`) is a wide operation that compares whole rows and shuffles both sides, so it is heavier than an explicit `is_valid` flag and behaves differently once you add a timestamp column.

**Example.** 10 million raw rows with 200 thousand bad ones: caching the checked frame and writing valid plus quarantine reads the source once at the cost of holding the frame; recomputing reads twice but needs no cache memory.

**Watch out for.** using `subtract` after adding a `rejected_at` timestamp. The columns no longer match, so no row cancels out and the quarantine gets everything. Derive a boolean and filter it twice.

## Choose built-in functions before a Python UDF

Use Spark built-in functions first. Python UDFs often move data between the JVM and Python worker, which can be slower and harder for Spark to optimize.

Prefer the built-in expression, and compare it with the Python UDF that does the same job:

In [ ]:
from pyspark.sql import types as T


@F.udf(T.StringType())
def email_domain_udf(email):
    if email is None or "@" not in email:
        return None
    return email.strip().lower().split("@")[-1]


builtin = raw_customers.withColumn(
    "email_domain",
    F.when(F.col("email").contains("@"), F.element_at(F.split(F.lower(F.trim("email")), "@"), -1)),
)
with_udf = raw_customers.withColumn("email_domain", email_domain_udf("email"))

assert sorted(map(tuple, builtin.collect())) == sorted(map(tuple, with_udf.collect()))
udf_plan = plan_of(with_udf)
assert "BatchEvalPython" in udf_plan and "BatchEvalPython" not in plan_of(builtin)
print("same answer; only the UDF plan crosses into Python (BatchEvalPython)")

If you need Python for vectorized numeric or ML work, a pandas UDF moves Arrow batches instead
of single rows (it needs `pandas` and `pyarrow` on every worker), but measure it:

```python
import pandas as pd
from pyspark.sql import functions as F, types as T

@F.pandas_udf(T.DoubleType())
def cap_score(score: pd.Series) -> pd.Series:
    return score.clip(lower=0.0, upper=1.0)

scored = users.withColumn("capped_score", cap_score("score"))
```

**How it works.** A Python UDF plans as a `BatchEvalPython` node: each task starts a Python worker, sends batches of rows over a socket, and receives results, usually via Arrow or pickles. Catalyst cannot push predicates through that boundary and whole-stage code generation stops there, so the UDF is a black box in the middle of an otherwise compiled pipeline.

**Example.** For 100 million rows, a row-at-a-time UDF costing one microsecond of Python per row adds about 100 seconds of interpreter time before you count serialization, and it blocks the generated code path around it.

**Watch out for.** an `@F.udf` closure that captures a large lookup dict or DataFrame. The object is pickled to every task; broadcast it as a Spark broadcast variable instead, or express the logic with built-ins.

## Make retries safe at the publish boundary

An **idempotent** write leaves the same logical result when you apply the same batch twice. A stable record key and deterministic input let a transactional table merge replace the previous version of a row. A bounded overwrite can also work when each batch fully rebuilds a known slice.

**Where it matters.** A job can commit its output and then fail before the scheduler records success. The scheduler retries the batch, so plain append can create duplicates.

**Watch out for.** Each merge key needs at most one source row per batch. Deduplicate using an ordering that survives replay. If older batches can arrive later, compare source versions before updating. Keep the original input available for reconciliation.

### Example

Suppose batch 42 contains order 101 at version 3 for 2500 cents. The first run inserts it and a retry leaves one row worth 2500 cents. Batch 43 updates order 101 to version 4 for 4200 cents. Replaying batch 42 after batch 43 should still leave one row at version 4. Use these three checks to test your merge before scheduling it.

## Check yourself

The lessons above end with takeaways. Re-run each cell in order, and use `explain` on any query you do not understand before you move on.

## Key takeaways

- Idempotent writes are the goal: rerunning a day must give the same table.
- Put business rules in functions and test them with a one-row DataFrame.
- Quarantine bad rows with a reason column instead of dropping them silently.

**Next:** [PySpark 6: structured streaming](pyspark-06-structured-streaming.ipynb)

## Run a version-aware merge on a local Delta table

Order 1 starts at source version 1. A correction at version 2 raises its amount from 10
to 15. Apply the correction twice and then replay the older delivery. The match predicate
identifies the order, and the update condition allows only a newer source version to win.

This lab uses the delta-rs Python API with a tiny Arrow fixture. Spark SQL needs a
compatible Delta connector to run equivalent table mutations. In production, validate
that each source batch contains at most one winning row per key before merging.

[Delta Python API](https://delta-io.github.io/delta-rs/api/delta_table/)

In [ ]:
import tempfile

import pyarrow as pa
from deltalake import DeltaTable, write_deltalake


def merge_orders(table, changes):
    (
        table.merge(changes, predicate="t.id = s.id", source_alias="s", target_alias="t")
        .when_matched_update_all(predicate="s.source_version > t.source_version")
        .when_not_matched_insert_all()
        .execute()
    )


with tempfile.TemporaryDirectory(prefix="delta-dml-") as directory:
    initial = pa.table({"id": [1, 2], "amount": [10, 20], "source_version": [1, 1]})
    write_deltalake(directory, initial)
    table = DeltaTable(directory)
    table.update(predicate="id = 1", updates={"amount": "amount + 2"})
    table.delete(predicate="id = 2")
    correction = pa.table({"id": [1, 3], "amount": [15, 30], "source_version": [2, 1]})

    merge_orders(table, correction)
    merge_orders(table, correction)
    stale_order = initial.slice(0, 1)
    merge_orders(table, stale_order)

    rows = sorted(table.to_pyarrow_table().to_pylist(), key=lambda row: row["id"])
    assert rows == [
        {"id": 1, "amount": 15, "source_version": 2},
        {"id": 3, "amount": 30, "source_version": 1},
    ]
    assert DeltaTable(directory, version=0).to_pyarrow_table().num_rows == 2
    print(rows)